In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean

In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
data = pd.read_excel('dataset_gen.xlsx') 
dataset_all_gen = pd.read_excel('dataset_all_gen.xlsx')
dataset_all_original = pd.read_excel('dataset_all_original.xlsx')
dataset_jaccard_gen = pd.read_excel('dataset_jaccard_gen.xlsx')
dataset_jaccard_original = pd.read_excel('dataset_jaccard_original.xlsx')
dataset_myroom_gen = pd.read_excel('dataset_myroom_gen.xlsx')
dataset_myroom_original = pd.read_excel('dataset_myroom_original.xlsx')
dataset_university_gen = pd.read_excel('dataset_university_gen.xlsx')
dataset_university_original = pd.read_excel('dataset_university_original.xlsx')

In [4]:
data = data.sample(frac = 1) 
dataset_all_gen = dataset_all_gen.sample(frac = 1) 
dataset_all_original = dataset_all_original.sample(frac = 1) 
dataset_jaccard_gen = dataset_jaccard_gen.sample(frac = 1) 
dataset_jaccard_original = dataset_jaccard_original.sample(frac = 1) 
dataset_myroom_gen = dataset_myroom_gen.sample(frac = 1) 
dataset_myroom_original = dataset_myroom_original.sample(frac = 1) 
dataset_university_gen = dataset_university_gen.sample(frac = 1) 
dataset_university_original = dataset_university_original.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)
dataset_all_gen = dataset_all_gen.reset_index(drop=True)
dataset_all_original = dataset_all_original.reset_index(drop=True)
dataset_jaccard_gen = dataset_jaccard_gen.reset_index(drop=True)
dataset_jaccard_original = dataset_jaccard_original.reset_index(drop=True)
dataset_myroom_gen = dataset_myroom_gen.reset_index(drop=True)
dataset_myroom_original = dataset_myroom_original.reset_index(drop=True)
dataset_university_gen = dataset_university_gen.reset_index(drop=True)
dataset_university_original = dataset_university_original.reset_index(drop=True)

In [6]:
data.isnull().sum()

type             0
Name             0
Vendor           0
DNS           2304
mDNS          2704
NUPNP        11000
open_port     1731
SSDP_Type     2304
SSDP_t       11000
SSDP_v       11000
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
#fill Na
dataset_all_gen = dataset_all_gen.fillna("NaN")
dataset_all_original = dataset_all_original.fillna("NaN")
dataset_jaccard_gen = dataset_jaccard_gen.fillna("NaN")
dataset_jaccard_original = dataset_jaccard_original.fillna("NaN")
dataset_myroom_gen = dataset_myroom_gen.fillna("NaN")
dataset_myroom_original = dataset_myroom_original.fillna("NaN")
dataset_university_gen = dataset_university_gen.fillna("NaN")
dataset_university_original = dataset_university_original.fillna("NaN")

In [9]:
data.isnull().sum()

type         0
Name         0
Vendor       0
DNS          0
mDNS         0
NUPNP        0
open_port    0
SSDP_Type    0
SSDP_t       0
SSDP_v       0
dtype: int64

In [10]:
data.dtypes

type         object
Name         object
Vendor       object
DNS          object
mDNS         object
NUPNP        object
open_port    object
SSDP_Type    object
SSDP_t       object
SSDP_v       object
dtype: object

In [11]:
data = data.astype('category')

In [12]:
dataset_all_gen = dataset_all_gen.astype('category')
dataset_all_original = dataset_all_original.astype('category')
dataset_jaccard_gen = dataset_jaccard_gen.astype('category')
dataset_jaccard_original = dataset_jaccard_original.astype('category')
dataset_myroom_gen = dataset_myroom_gen.astype('category')
dataset_myroom_original = dataset_myroom_original.astype('category')
dataset_university_gen = dataset_university_gen.astype('category')
dataset_university_original = dataset_university_original.astype('category')

In [13]:
data.dtypes

type         category
Name         category
Vendor       category
DNS          category
mDNS         category
NUPNP        category
open_port    category
SSDP_Type    category
SSDP_t       category
SSDP_v       category
dtype: object

In [14]:
data

,type,Name,Vendor,DNS,mDNS,NUPNP,open_port,SSDP_Type,SSDP_t,SSDP_v
0,Appliances,Bose SoundTouch 10,Bose Corporation,buzzfeed.com,NaN,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
1,Appliances,3X Google Home Mini,Google,blogtalkradio.com,_viva._udp.local.,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
2,Hubs,Securifi Almond,Securifi,wufoo.com,_biodex._udp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,NaN,NaN
3,Hubs,Apple HomePod,"Apple, Inc.",NaN,_cookley._udp.local.,NaN,tcp/1900,NaN,NaN,NaN
4,Appliances,iRobot,iRobot Corporation,domainmarket.com,_domainer._tcp.local.,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
5,Appliances,Triby Speaker Smart Speaker,Invoxia,sourceforge.net,NaN,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
6,Hubs,Philips HUE Hub,Philips Lighting BV,youtu.be,NaN,NaN,NaN,NOTIFY * HTTP/1.1,NaN,NaN
7,Non-IOT,Lg Smartphone,LG Electronics,nps.gov,NaN,NaN,tcp/1900,M-SEARCH * HTTP/1.1,NaN,NaN
8,Appliances,Bose SoundTouch 10,Bose Corporation,biblegateway.com,_keylex._tcp.local.,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
9,Hubs,Logitech Harmony Hub,Logitech,oracle.com,_toughjoyfax._tcp.local.,NaN,tcp/22,NOTIFY * HTTP/1.1,NaN,NaN


In [15]:
y = data.drop_duplicates(keep='first')
y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 


In [16]:
len(y)

10610

In [17]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original.drop_duplicates(keep='first')

In [18]:
#drop name
dataset_all_gen_y = dataset_all_gen_y.drop(['Name'], axis=1)
dataset_all_original_y = dataset_all_original_y.drop(['Name'], axis=1)
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop(['Name'], axis=1)
dataset_jaccard_original_y = dataset_jaccard_original_y.drop(['Name'], axis=1)
dataset_myroom_gen_y = dataset_myroom_gen_y.drop(['Name'], axis=1)
dataset_myroom_original_y = dataset_myroom_original_y.drop(['Name'], axis=1)
dataset_university_gen_y = dataset_university_gen_y.drop(['Name'], axis=1)
dataset_university_original_y = dataset_university_original_y.drop(['Name'], axis=1)

In [19]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen_y.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original_y.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original_y.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen_y.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original_y.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen_y.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original_y.drop_duplicates(keep='first')

In [20]:
y = y.apply(LabelEncoder().fit_transform)

In [21]:
dataset_all_gen_y = dataset_all_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_all_original_y = dataset_all_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_original_y = dataset_jaccard_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_gen_y = dataset_myroom_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_original_y = dataset_myroom_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_gen_y = dataset_university_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_original_y = dataset_university_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

In [22]:
list_data = []
list_data.append(y)
list_data.append(dataset_all_gen_y)
list_data.append(dataset_all_original_y)
list_data.append(dataset_jaccard_gen_y)
list_data.append(dataset_jaccard_original_y)
list_data.append(dataset_myroom_gen_y)
list_data.append(dataset_myroom_original_y)
list_data.append(dataset_university_gen_y)
list_data.append(dataset_university_original_y)
len(list_data)

9

In [23]:
accuracy = []
accuracy_t = []
accuracy_temp = []

In [24]:
for data in list_data:
    for i in range(100):
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
#         kf.get_n_splits(data)
        for train_index, test_index in kf.split(data):
        #     print(index)
#             print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
#             print("TRAIN:", train, "TEST:", test)
        #     print(len(train))
#             print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index(drop=True)
            clf = tree.DecisionTreeClassifier()
            clf = clf.fit(train_data,label_data )
            predict = clf.predict(test_data)
#             print(truelabel_data)
#             print(predict)
        #     len(predict)
            result_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
#             print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            print(i,"  Accuracy: ",t)
        accuracy_t.append(mean(accuracy_temp))
    accuracy.append(mean(accuracy_t))

0   Accuracy:  0.8397737983034873
0   Accuracy:  0.8397737983034873
0   Accuracy:  0.8491988689915174
0   Accuracy:  0.827521206409048
0   Accuracy:  0.8529688972667295
0   Accuracy:  0.8341187558906692
0   Accuracy:  0.8369462770970783
0   Accuracy:  0.8426013195098964
0   Accuracy:  0.8331762488218661
0   Accuracy:  0.8510838831291234
1   Accuracy:  0.8303487276154571
1   Accuracy:  0.8454288407163054
1   Accuracy:  0.8426013195098964
1   Accuracy:  0.8303487276154571
1   Accuracy:  0.8482563619227145
1   Accuracy:  0.8303487276154571
1   Accuracy:  0.824693685202639
1   Accuracy:  0.8303487276154571
1   Accuracy:  0.8303487276154571
1   Accuracy:  0.8661639962299718
2   Accuracy:  0.8397737983034873
2   Accuracy:  0.825636192271442
2   Accuracy:  0.8473138548539114
2   Accuracy:  0.8595664467483506
2   Accuracy:  0.8426013195098964
2   Accuracy:  0.8350612629594723
2   Accuracy:  0.8463713477851084
2   Accuracy:  0.8586239396795476
2   Accuracy:  0.8303487276154571
2   Accuracy:  0.

24   Accuracy:  0.8605089538171536
24   Accuracy:  0.8265786993402451
24   Accuracy:  0.8322337417530632
24   Accuracy:  0.8520263901979265
24   Accuracy:  0.8265786993402451
24   Accuracy:  0.8473138548539114
24   Accuracy:  0.8529688972667295
25   Accuracy:  0.8312912346842601
25   Accuracy:  0.8482563619227145
25   Accuracy:  0.8378887841658812
25   Accuracy:  0.8397737983034873
25   Accuracy:  0.8407163053722903
25   Accuracy:  0.8312912346842601
25   Accuracy:  0.8284637134778511
25   Accuracy:  0.8463713477851084
25   Accuracy:  0.8557964184731386
25   Accuracy:  0.8265786993402451
26   Accuracy:  0.8416588124410933
26   Accuracy:  0.8284637134778511
26   Accuracy:  0.8557964184731386
26   Accuracy:  0.8454288407163054
26   Accuracy:  0.8510838831291234
26   Accuracy:  0.8576814326107446
26   Accuracy:  0.8162111215834119
26   Accuracy:  0.820923656927427
26   Accuracy:  0.823751178133836
26   Accuracy:  0.8482563619227145
27   Accuracy:  0.8548539114043355
27   Accuracy:  0.8303

47   Accuracy:  0.8463713477851084
48   Accuracy:  0.824693685202639
48   Accuracy:  0.8378887841658812
48   Accuracy:  0.8303487276154571
48   Accuracy:  0.8520263901979265
48   Accuracy:  0.8397737983034873
48   Accuracy:  0.8397737983034873
48   Accuracy:  0.8416588124410933
48   Accuracy:  0.8614514608859567
48   Accuracy:  0.8341187558906692
48   Accuracy:  0.8567389255419415
49   Accuracy:  0.8416588124410933
49   Accuracy:  0.8444863336475024
49   Accuracy:  0.8529688972667295
49   Accuracy:  0.8312912346842601
49   Accuracy:  0.8284637134778511
49   Accuracy:  0.8397737983034873
49   Accuracy:  0.8331762488218661
49   Accuracy:  0.8539114043355325
49   Accuracy:  0.8397737983034873
49   Accuracy:  0.8491988689915174
50   Accuracy:  0.8426013195098964
50   Accuracy:  0.8378887841658812
50   Accuracy:  0.8501413760603205
50   Accuracy:  0.8426013195098964
50   Accuracy:  0.8501413760603205
50   Accuracy:  0.82186616399623
50   Accuracy:  0.8501413760603205
50   Accuracy:  0.84919

71   Accuracy:  0.8388312912346843
71   Accuracy:  0.8557964184731386
71   Accuracy:  0.8501413760603205
72   Accuracy:  0.8369462770970783
72   Accuracy:  0.8350612629594723
72   Accuracy:  0.8463713477851084
72   Accuracy:  0.8501413760603205
72   Accuracy:  0.8303487276154571
72   Accuracy:  0.8463713477851084
72   Accuracy:  0.8303487276154571
72   Accuracy:  0.8416588124410933
72   Accuracy:  0.8435438265786993
72   Accuracy:  0.8407163053722903
73   Accuracy:  0.8388312912346843
73   Accuracy:  0.8378887841658812
73   Accuracy:  0.824693685202639
73   Accuracy:  0.8350612629594723
73   Accuracy:  0.8322337417530632
73   Accuracy:  0.8416588124410933
73   Accuracy:  0.8407163053722903
73   Accuracy:  0.8435438265786993
73   Accuracy:  0.8388312912346843
73   Accuracy:  0.8435438265786993
74   Accuracy:  0.8510838831291234
74   Accuracy:  0.8510838831291234
74   Accuracy:  0.8199811498586239
74   Accuracy:  0.8143261074458058
74   Accuracy:  0.8284637134778511
74   Accuracy:  0.862

95   Accuracy:  0.8454288407163054
95   Accuracy:  0.8322337417530632
95   Accuracy:  0.8388312912346843
96   Accuracy:  0.8322337417530632
96   Accuracy:  0.8369462770970783
96   Accuracy:  0.8501413760603205
96   Accuracy:  0.8426013195098964
96   Accuracy:  0.825636192271442
96   Accuracy:  0.8491988689915174
96   Accuracy:  0.8454288407163054
96   Accuracy:  0.8416588124410933
96   Accuracy:  0.8331762488218661
96   Accuracy:  0.8548539114043355
97   Accuracy:  0.8473138548539114
97   Accuracy:  0.8378887841658812
97   Accuracy:  0.823751178133836
97   Accuracy:  0.8501413760603205
97   Accuracy:  0.8454288407163054
97   Accuracy:  0.8322337417530632
97   Accuracy:  0.8416588124410933
97   Accuracy:  0.8369462770970783
97   Accuracy:  0.8407163053722903
97   Accuracy:  0.8303487276154571
98   Accuracy:  0.8444863336475024
98   Accuracy:  0.8294062205466541
98   Accuracy:  0.8539114043355325
98   Accuracy:  0.8463713477851084
98   Accuracy:  0.8360037700282752
98   Accuracy:  0.8341

19   Accuracy:  0.8439716312056738
19   Accuracy:  0.8359929078014184
19   Accuracy:  0.8413120567375887
20   Accuracy:  0.8529672276350753
20   Accuracy:  0.8529672276350753
20   Accuracy:  0.829052258635961
20   Accuracy:  0.8511957484499557
20   Accuracy:  0.8733392382639504
20   Accuracy:  0.8379096545615589
20   Accuracy:  0.8653675819309123
20   Accuracy:  0.8351063829787234
20   Accuracy:  0.8572695035460993
20   Accuracy:  0.8315602836879432
21   Accuracy:  0.8804251550044287
21   Accuracy:  0.8432240921169176
21   Accuracy:  0.8405668733392383
21   Accuracy:  0.8387953941541186
21   Accuracy:  0.8343666961913198
21   Accuracy:  0.8414526129317981
21   Accuracy:  0.8379096545615589
21   Accuracy:  0.8581560283687943
21   Accuracy:  0.850177304964539
21   Accuracy:  0.8546099290780141
22   Accuracy:  0.8352524357838795
22   Accuracy:  0.870682019486271
22   Accuracy:  0.8547387068201948
22   Accuracy:  0.8618246235606731
22   Accuracy:  0.8423383525243578
22   Accuracy:  0.85562

... (24 outputs left out)

11   Accuracy:  1.0
11   Accuracy:  1.0
11   Accuracy:  1.0
11   Accuracy:  1.0
11   Accuracy:  0.9285714285714286
11   Accuracy:  0.8928571428571429
11   Accuracy:  0.8928571428571429
11   Accuracy:  0.9285714285714286
11   Accuracy:  0.8928571428571429
11   Accuracy:  0.9642857142857143
12   Accuracy:  0.9655172413793104
12   Accuracy:  0.9655172413793104
12   Accuracy:  0.9655172413793104
12   Accuracy:  1.0
12   Accuracy:  0.9642857142857143
12   Accuracy:  0.9285714285714286
12   Accuracy:  0.9642857142857143
12   Accuracy:  0.9285714285714286
12   Accuracy:  1.0
12   Accuracy:  0.8928571428571429
13   Accuracy:  0.9310344827586207
13   Accuracy:  0.896551724137931
13   Accuracy:  1.0
13   Accuracy:  0.9655172413793104
13   Accuracy:  0.9642857142857143
13   Accuracy:  1.0
13   Accuracy:  0.9642857142857143
13   Accuracy:  0.9642857142857143
13   Accuracy:  0.9642857142857143
13   Accuracy:  0.9285714285714286
14   Accuracy:  0.9310344827586207
14   Accuracy:  0.896551724137931
14

37   Accuracy:  1.0
37   Accuracy:  0.9642857142857143
37   Accuracy:  0.8571428571428571
37   Accuracy:  1.0
37   Accuracy:  0.9642857142857143
38   Accuracy:  0.9310344827586207
38   Accuracy:  0.9310344827586207
38   Accuracy:  0.9310344827586207
38   Accuracy:  1.0
38   Accuracy:  0.9642857142857143
38   Accuracy:  1.0
38   Accuracy:  0.9642857142857143
38   Accuracy:  1.0
38   Accuracy:  0.9642857142857143
38   Accuracy:  0.9642857142857143
39   Accuracy:  0.896551724137931
39   Accuracy:  0.9310344827586207
39   Accuracy:  0.9655172413793104
39   Accuracy:  0.9310344827586207
39   Accuracy:  1.0
39   Accuracy:  0.9642857142857143
39   Accuracy:  1.0
39   Accuracy:  0.9285714285714286
39   Accuracy:  0.9642857142857143
39   Accuracy:  1.0
40   Accuracy:  0.9655172413793104
40   Accuracy:  1.0
40   Accuracy:  0.9655172413793104
40   Accuracy:  0.9655172413793104
40   Accuracy:  0.8214285714285714
40   Accuracy:  0.9285714285714286
40   Accuracy:  0.9285714285714286
40   Accuracy:  

64   Accuracy:  0.9642857142857143
64   Accuracy:  0.9285714285714286
64   Accuracy:  1.0
64   Accuracy:  0.9642857142857143
65   Accuracy:  1.0
65   Accuracy:  0.8275862068965517
65   Accuracy:  0.9655172413793104
65   Accuracy:  0.9310344827586207
65   Accuracy:  0.9642857142857143
65   Accuracy:  0.9642857142857143
65   Accuracy:  0.8928571428571429
65   Accuracy:  0.9642857142857143
65   Accuracy:  0.9642857142857143
65   Accuracy:  1.0
66   Accuracy:  0.9655172413793104
66   Accuracy:  0.9310344827586207
66   Accuracy:  1.0
66   Accuracy:  0.9655172413793104
66   Accuracy:  0.8928571428571429
66   Accuracy:  0.8928571428571429
66   Accuracy:  1.0
66   Accuracy:  1.0
66   Accuracy:  0.8571428571428571
66   Accuracy:  0.9285714285714286
67   Accuracy:  1.0
67   Accuracy:  0.896551724137931
67   Accuracy:  0.9310344827586207
67   Accuracy:  0.9655172413793104
67   Accuracy:  0.8928571428571429
67   Accuracy:  0.9642857142857143
67   Accuracy:  0.9642857142857143
67   Accuracy:  0.964

91   Accuracy:  0.9642857142857143
91   Accuracy:  1.0
91   Accuracy:  0.9285714285714286
91   Accuracy:  0.8571428571428571
92   Accuracy:  0.896551724137931
92   Accuracy:  1.0
92   Accuracy:  1.0
92   Accuracy:  0.9655172413793104
92   Accuracy:  0.9642857142857143
92   Accuracy:  1.0
92   Accuracy:  0.9642857142857143
92   Accuracy:  1.0
92   Accuracy:  0.9642857142857143
92   Accuracy:  0.8928571428571429
93   Accuracy:  0.896551724137931
93   Accuracy:  0.9655172413793104
93   Accuracy:  0.9655172413793104
93   Accuracy:  1.0
93   Accuracy:  1.0
93   Accuracy:  0.9642857142857143
93   Accuracy:  0.9285714285714286
93   Accuracy:  0.9642857142857143
93   Accuracy:  0.8214285714285714
93   Accuracy:  1.0
94   Accuracy:  0.9310344827586207
94   Accuracy:  1.0
94   Accuracy:  0.9655172413793104
94   Accuracy:  0.9310344827586207
94   Accuracy:  1.0
94   Accuracy:  0.9642857142857143
94   Accuracy:  0.9642857142857143
94   Accuracy:  0.8928571428571429
94   Accuracy:  0.92857142857142

In [25]:
print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[2])
print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[4])
print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[6])
print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[8])

10610 	 Accuracy for dataset_gen: 	 0.8399380239487126
11287 	 Accuracy for dataset_all_gen: 	 0.8411195293428186
677 	 Accuracy for dataset_all_original: 	 0.8506168993983667
10912 	 Accuracy for dataset_jaccard_gen: 	 0.858655981353084
302 	 Accuracy for dataset_jaccard_original: 	 0.8651003827575833
10701 	 Accuracy for dataset_myroom_gen: 	 0.8703312436470372
91 	 Accuracy for dataset_myroom_original: 	 0.8745679997119024
10894 	 Accuracy for dataset_university_gen: 	 0.878126967490074
284 	 Accuracy for dataset_university_original: 	 0.8808520111161877
